# Phase 02 — Data Loading, Quality & Cleaning
## Rappi Delivery Operations Dataset

**Purpose:** Produce validated, typed, and cleaned base datasets for all downstream analysis.

**Outputs:**
- `outputs/cleaned/raw_data_clean.parquet`
- `outputs/cleaned/zone_info_clean.parquet`
- `outputs/cleaned/zone_polygons_clean.parquet`

> This notebook does **not** build KPIs or draw business conclusions.  
> Downstream notebooks (`02`, `03`, `04`) load from the parquet files produced here.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.ticker as mticker

from src.config import (
    DATA_PATH,
    RAW_CLEAN_PATH, ZONE_INFO_CLEAN_PATH, POLYGONS_CLEAN_PATH,
)
from src.io_utils import load_all_sheets, save_parquet
from src.cleaning import standardize_types, add_time_features, clean_text_key, flag_invalid_rows, standardize_lookup
from src.validation import check_grain, check_key_uniqueness, compare_zone_sets, validate_merge_cardinality

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("Libraries loaded.")


Libraries loaded.


---
## A. Reproducible Loading

In [2]:
# Load all sheets
sheets = load_all_sheets(DATA_PATH)

print(f"Sheets found ({len(sheets)}): {list(sheets.keys())}")
for name, df in sheets.items():
    print(f"  {name:25s}  {df.shape[0]:6,} rows  ×  {df.shape[1]} cols  |  {list(df.columns)}")

# ── Assign roles based on what we saw in Phase 00 ──────────────────────────
sheet_names = list(sheets.keys())

raw_sheet_name       = sheet_names[0]
zone_info_sheet_name = sheet_names[1]
polygons_sheet_name  = sheet_names[2]

raw       = sheets[raw_sheet_name].copy()
zone_info = sheets[zone_info_sheet_name].copy()
polygons  = sheets[polygons_sheet_name].copy()


Sheets found (3): ['RAW_DATA', 'ZONE_INFO', 'ZONE_POLYGONS']
  RAW_DATA                   10,080 rows  ×  9 cols  |  ['COUNTRY', 'DATE', 'HOUR', 'CITY', 'ZONE', 'CONNECTED_RT', 'ORDERS', 'EARNINGS', 'PRECIPITATION_MM']
  ZONE_INFO                      14 rows  ×  4 cols  |  ['ZONE', 'LATITUDE_CENTER', 'LONGITUDE_CENTER', 'DESCRIPTION']
  ZONE_POLYGONS                  14 rows  ×  3 cols  |  ['ZONE_ID', 'ZONE_NAME', 'GEOMETRY_WKT']


---
## B. Type Conversion & Derived Features

In [3]:
raw = standardize_types(raw)
raw = add_time_features(raw)

zone_info = standardize_types(zone_info)
polygons  = standardize_types(polygons)

print("RAW_DATA dtypes after standardisation:")
print(raw.dtypes)
print(f"\nDate range: {raw['DATE'].min().date()} → {raw['DATE'].max().date()}")
print(f"HOUR range: {raw['HOUR'].min()} – {raw['HOUR'].max()}")

RAW_DATA dtypes after standardisation:
COUNTRY                        str
DATE                datetime64[us]
HOUR                         Int64
CITY                           str
ZONE                           str
CONNECTED_RT                 int64
ORDERS                       int64
EARNINGS                   float64
PRECIPITATION_MM           float64
DAY_OF_WEEK                  int32
DAY_NAME                       str
MONTH                        int32
IS_WEEKEND                    bool
DATETIME            datetime64[us]
dtype: object

Date range: 2024-03-01 → 2024-03-30
HOUR range: 0 – 23


---
## C. Grain Validation

Test whether `COUNTRY + DATE + HOUR + CITY + ZONE` uniquely identifies every row in `RAW_DATA`.  
This is a critical structural check before any merge or aggregation.

In [4]:
GRAIN_KEYS = ["COUNTRY", "DATE", "HOUR", "CITY", "ZONE"]
grain_dupes = check_grain(raw, GRAIN_KEYS)

if not grain_dupes.empty:
    print(f"\nSample duplicate rows:")
    print(grain_dupes.head(10).to_string(index=False))
    print(f"\n→ Review duplicates before deciding to deduplicate or aggregate.")
else:
    print("Grain is valid — safe to merge and aggregate.")

✓ Grain valid — all 5-key combinations are unique.
Grain is valid — safe to merge and aggregate.


---
## D. Missing Value Analysis

In [5]:
def missing_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Produce a per-column missingness summary."""
    miss = df.isna().sum().reset_index()
    miss.columns = ["Column", "Missing_Count"]
    miss["Missing_Pct"] = (miss["Missing_Count"] / len(df) * 100).round(2)
    miss["Table"] = name
    return miss[miss["Missing_Count"] > 0].sort_values("Missing_Pct", ascending=False)

miss_raw  = missing_report(raw,       "RAW_DATA")
miss_info = missing_report(zone_info, "ZONE_INFO")
miss_poly = missing_report(polygons,  "ZONE_POLYGONS")

all_missing = pd.concat([miss_raw, miss_info, miss_poly], ignore_index=True)

if all_missing.empty:
    print("✓ No missing values found in any sheet.")
else:
    print(f"Missing value report ({len(all_missing)} columns affected):")
    print(all_missing.to_string(index=False))

✓ No missing values found in any sheet.


---
## E. Invalid Value Checks

Data might be valid but it can be out of valid ranges

In [6]:
raw = flag_invalid_rows(raw)

n_invalid = raw["IS_INVALID"].sum()
print(f"Invalid rows flagged: {n_invalid:,} ({n_invalid/len(raw):.2%} of total)")

if n_invalid > 0:
    print("\nBreakdown by reason:")
    reasons = (
        raw.loc[raw["IS_INVALID"], "INVALID_REASON"]
        .str.split(";").explode()
        .value_counts()
    )
    print(reasons.to_string())
    print("\nSample invalid rows:")
    print(raw[raw["IS_INVALID"]].head(5)[GRAIN_KEYS + ["CONNECTED_RT","ORDERS","EARNINGS","PRECIPITATION_MM","INVALID_REASON"]].to_string(index=False))

Invalid rows flagged: 0 (0.00% of total)


---
## F. Consistency Checks

In [7]:
# 1. Zone → city mapping consistency (one zone should belong to one city)
if "CITY" in raw.columns and "ZONE" in raw.columns:
    zone_city_map = raw.groupby("ZONE")["CITY"].nunique()
    multi_city_zones = zone_city_map[zone_city_map > 1]
    if multi_city_zones.empty:
        print("✓ Every zone maps to exactly one city.")
    else:
        print("✗ Zones mapping to multiple cities:")
        print(multi_city_zones)

# 2. Date range continuity
if "DATE" in raw.columns:
    all_dates = pd.date_range(raw["DATE"].min(), raw["DATE"].max(), freq="D")
    present   = raw["DATE"].dt.normalize().unique()
    missing_dates = set(all_dates) - set(present)
    if missing_dates:
        print(f"\n✗ {len(missing_dates)} missing date(s): {sorted(d.date() for d in missing_dates)}")
    else:
        print(f"✓ Date coverage is continuous: {raw['DATE'].min().date()} → {raw['DATE'].max().date()}")

# 3. Hour coverage per date (detect systematically missing hours)
if "HOUR" in raw.columns:
    hours_per_day = raw.groupby("DATE")["HOUR"].nunique()
    unusual = hours_per_day[hours_per_day < 20]
    if unusual.empty:
        print(f"✓ All dates have ≥ 20 hours covered.")
    else:
        print(f"✗ Dates with fewer than 20 hours: {len(unusual)}")
        print(unusual.sort_values().head(10))

✓ Every zone maps to exactly one city.
✓ Date coverage is continuous: 2024-03-01 → 2024-03-30
✓ All dates have ≥ 20 hours covered.


---
## G. Lookup Table Validation

Validate uniqueness of join keys and zone coverage across all three sheets.

In [8]:
# Key uniqueness checks on dimension tables
print("=== ZONE_INFO: ZONE uniqueness ===")
_ = check_key_uniqueness(zone_info, "ZONE")

poly_name_col = "ZONE_NAME" if "ZONE_NAME" in polygons.columns else polygons.columns[1]
print(f"\n=== ZONE_POLYGONS: {poly_name_col} uniqueness ===")
_ = check_key_uniqueness(polygons, poly_name_col)

if "ZONE_ID" in polygons.columns:
    print(f"\n=== ZONE_POLYGONS: ZONE_ID uniqueness ===")
    _ = check_key_uniqueness(polygons, "ZONE_ID")

=== ZONE_INFO: ZONE uniqueness ===
✓ 'ZONE' is unique — no duplicate values found.

=== ZONE_POLYGONS: ZONE_NAME uniqueness ===
✓ 'ZONE_NAME' is unique — no duplicate values found.

=== ZONE_POLYGONS: ZONE_ID uniqueness ===
✓ 'ZONE_ID' is unique — no duplicate values found.


In [9]:
# Standardise join keys before comparing zones
raw       = standardize_lookup(raw,       ["ZONE"])
zone_info = standardize_lookup(zone_info, ["ZONE"])
polygons  = standardize_lookup(polygons,  [poly_name_col])

# Rename to the shared key name used across all three tables
if f"{poly_name_col}_CLEAN" in polygons.columns:
    polygons = polygons.rename(columns={f"{poly_name_col}_CLEAN": "ZONE_CLEAN"})

# Compare zone sets
print("\n=== Zone coverage comparison (using normalised keys) ===")
coverage = compare_zone_sets(raw, zone_info, polygons)



=== Zone coverage comparison (using normalised keys) ===
RAW_DATA zones     : 14
ZONE_INFO zones    : 14
ZONE_POLYGONS zones: 14
In RAW not in ZONE_INFO    : set()
In RAW not in POLYGONS     : set()
Orphans in ZONE_INFO       : set()
Orphans in ZONE_POLYGONS   : set()
Full match across all sheets: True


In [10]:
raw.columns

Index(['COUNTRY', 'DATE', 'HOUR', 'CITY', 'ZONE', 'CONNECTED_RT', 'ORDERS', 'EARNINGS', 'PRECIPITATION_MM',
       'DAY_OF_WEEK', 'DAY_NAME', 'MONTH', 'IS_WEEKEND', 'DATETIME', 'IS_INVALID', 'INVALID_REASON', 'ZONE_CLEAN'],
      dtype='str')

In [11]:
polygons.columns

Index(['ZONE_ID', 'ZONE_NAME', 'GEOMETRY_WKT', 'ZONE_CLEAN'], dtype='str')

In [12]:
# Merge cardinality validation before any join
print("=== Cardinality: RAW_DATA → ZONE_INFO ===")
validate_merge_cardinality(raw, zone_info, "ZONE_CLEAN")

print("\n=== Cardinality: RAW_DATA → ZONE_POLYGONS ===")
validate_merge_cardinality(raw, polygons, "ZONE_CLEAN")

=== Cardinality: RAW_DATA → ZONE_INFO ===
Left  'ZONE_CLEAN' unique: False  (14 unique / 10080 rows)
Right 'ZONE_CLEAN' unique: True  (14 unique / 14 rows)
→ Merge relationship: many-to-one

=== Cardinality: RAW_DATA → ZONE_POLYGONS ===
Left  'ZONE_CLEAN' unique: False  (14 unique / 10080 rows)
Right 'ZONE_CLEAN' unique: True  (14 unique / 14 rows)
→ Merge relationship: many-to-one


'many-to-one'

---
## H. Save Cleaned Outputs

In [13]:
save_parquet(raw,       RAW_CLEAN_PATH)
save_parquet(zone_info, ZONE_INFO_CLEAN_PATH)
save_parquet(polygons,  POLYGONS_CLEAN_PATH)

print("\nAll cleaned files saved. Notebook 02 and 03 can now load from outputs/cleaned/")

Saved parquet → outputs\cleaned\raw_data_clean.parquet  (10,080 rows)
Saved parquet → outputs\cleaned\zone_info_clean.parquet  (14 rows)
Saved parquet → outputs\cleaned\zone_polygons_clean.parquet  (14 rows)

All cleaned files saved. Notebook 02 and 03 can now load from outputs/cleaned/


---
## I. Quality Report Summary

| Check | Result | Severity | Action |
|---|---|---|---|
| Grain uniqueness | See cell output above | High | Deduplicate if needed |
| Missing values | See cell output above | Medium | Flag/impute key fields |
| Invalid values (HOUR, negatives, blanks) | See cell output above | High | Keep flagged; exclude from KPI calcs |
| Date continuity | See cell output above | Medium | Document gaps |
| Zone-city consistency | See cell output above | High | Fix if multi-city zones found |
| ZONE key uniqueness (ZONE_INFO) | See cell output above | High | Must be unique before merge |
| ZONE_NAME uniqueness (ZONE_POLYGONS) | See cell output above | High | Must be unique before merge |
| Zone coverage match | See cell output above | Medium | Document unmatched zones |
| Merge cardinality | See cell output above | High | Stop if many-to-many detected |

### Merge-readiness verdict
> If all cardinality checks passed and zone sets are aligned, the cleaned tables are ready for merging in downstream notebooks.

> **Next step:** Run `02_exploratory_analysis.ipynb`